In [23]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [24]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

simulator = AerSimulator()

In [25]:
# Quantum random bit generator

def quantum_random_bits(n: int) -> list[int]:
    """
    Generate n random bits by measuring n qubits each prepared in |+> = H|0>.
    Returns a list of 0s and 1s.
    """
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)          # |0> -> |+> = 1/sqrt(2)(|0>+|1>)
    qc.measure(range(n), range(n))

    job = simulator.run(qc, shots=1, memory=True)
    result = job.result()
    bitstring = result.get_memory()[0]  # e.g. '01101...'

    # Qiskit returns bits in reverse order (qubit 0 is rightmost)
    return [int(b) for b in reversed(bitstring)]

# Quick sanity check
test_bits = quantum_random_bits(10)
print("Sample quantum random bits:", test_bits)

Sample quantum random bits: [0, 0, 0, 1, 1, 1, 0, 0, 0, 0]


In [26]:
NUM_QUBITS = 100  # Total qubits sent; more = larger final key

# Alice prepare qubits

alice_bits  = quantum_random_bits(NUM_QUBITS)   # Secret bit values
alice_bases = quantum_random_bits(NUM_QUBITS)   # 0=Z-basis, 1=X-basis

def alice_prepare_qubit(bit: int, basis: int) -> QuantumCircuit:
    """
    Alice prepares a single qubit encoding `bit` in `basis`.
      basis=0 (Z): |0> or |1>
      basis=1 (X): |+> or |->
    Returns a QuantumCircuit with 1 qubit (no measurement — Bob will measure).
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)          # Flip to |1>
    if basis == 1:
        qc.h(0)          # Z-basis -> X-basis
    return qc

alice_circuits = [
    alice_prepare_qubit(alice_bits[i], alice_bases[i])
    for i in range(NUM_QUBITS)
]

print(f"Alice prepared {NUM_QUBITS} qubits.")
print("Alice bits  (first 20):", alice_bits[:20])
print("Alice bases (first 20):", alice_bases[:20], "  (0=Z, 1=X)")

Alice prepared 100 qubits.
Alice bits  (first 20): [0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1]
Alice bases (first 20): [0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1]   (0=Z, 1=X)


In [27]:
# Bob measure qubits

bob_bases = quantum_random_bits(NUM_QUBITS)   # 0=Z-basis, 1=X-basis

def bob_measure(alice_circuit: QuantumCircuit, basis: int) -> int:
    """
    Bob appends his measurement in `basis` to Alice's circuit and runs it.
    Returns 0 or 1.
    """
    qc = alice_circuit.copy()
    if basis == 1:
        qc.h(0)          # Rotate X-basis back to Z before measuring
    qc.measure(0, 0)

    job = simulator.run(qc, shots=1, memory=True)
    return int(job.result().get_memory()[0])

bob_results = [
    bob_measure(alice_circuits[i], bob_bases[i])
    for i in range(NUM_QUBITS)
]

print("Bob bases   (first 20):", bob_bases[:20], "  (0=Z, 1=X)")
print("Bob results (first 20):", bob_results[:20])

Bob bases   (first 20): [0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0]   (0=Z, 1=X)
Bob results (first 20): [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0]


In [28]:
# Public Channel: Basis Comparison
matching_indices = [
    i for i in range(NUM_QUBITS)
    if alice_bases[i] == bob_bases[i]
]

alice_sifted_key = [alice_bits[i]    for i in matching_indices]
bob_sifted_key   = [bob_results[i]   for i in matching_indices]

print(f"Qubits sent          : {NUM_QUBITS}")
print(f"Matching bases       : {len(matching_indices)}  "
      f"({100*len(matching_indices)/NUM_QUBITS:.1f}% — expected ~50%)")
print(f"\nAlice sifted key (first 20): {alice_sifted_key[:20]}")
print(f"  Bob sifted key (first 20): {bob_sifted_key[:20]}")

Qubits sent          : 100
Matching bases       : 59  (59.0% — expected ~50%)

Alice sifted key (first 20): [0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1]
  Bob sifted key (first 20): [0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1]


In [29]:
# Error Estimation and Attack Detection

SAMPLE_FRACTION = 0.25   # Use 25% of sifted key for error checking
QBER_THRESHOLD  = 0.11   # Abort if QBER > 11%

sifted_len   = len(alice_sifted_key)
sample_size  = max(1, int(sifted_len * SAMPLE_FRACTION))

# Choose sample positions using quantum randomness
sample_rng   = quantum_random_bits(sifted_len)  # One random bit per position

# Take the first `sample_size` positions where the random bit was 1
sample_positions = [i for i, r in enumerate(sample_rng) if r == 1][:sample_size]

if len(sample_positions) < sample_size:
    # Fall back to deterministic fill if quantum RNG gave too few 1s
    remaining = [i for i in range(sifted_len) if i not in sample_positions]
    sample_positions += remaining[:sample_size - len(sample_positions)]
sample_positions = sorted(sample_positions[:sample_size])

errors = sum(
    1 for i in sample_positions
    if alice_sifted_key[i] != bob_sifted_key[i]
)
qber = errors / sample_size

print(f"Sample size : {sample_size} bits")
print(f"Errors found: {errors}")
print(f"QBER        : {qber:.2%}")

if qber > QBER_THRESHOLD:
    print(f"\nATTACK DETECTED! QBER {qber:.2%} exceeds threshold {QBER_THRESHOLD:.0%}. Protocol aborted.")
else:
    print(f"\nNo attack detected (QBER {qber:.2%} ≤ threshold {QBER_THRESHOLD:.0%}).")

Sample size : 14 bits
Errors found: 0
QBER        : 0.00%

No attack detected (QBER 0.00% ≤ threshold 11%).


In [30]:
# Final Shared Key

sample_set = set(sample_positions)
final_key_indices = [i for i in range(sifted_len) if i not in sample_set]

alice_final_key = [alice_sifted_key[i] for i in final_key_indices]
bob_final_key   = [bob_sifted_key[i]   for i in final_key_indices]

keys_match = alice_final_key == bob_final_key

print(f"Final key length : {len(alice_final_key)} bits")
print(f"Keys identical   : {keys_match}")
print(f"\nAlice final key (first 30): {alice_final_key[:30]}")
print(f"  Bob final key (first 30): {bob_final_key[:30]}")

if keys_match and qber <= QBER_THRESHOLD:
    print("\nBB84 protocol complete — Alice and Bob share a secret key!")
else:
    print("\nKey mismatch or protocol aborted.")

Final key length : 45 bits
Keys identical   : True

Alice final key (first 30): [0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1]
  Bob final key (first 30): [0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1]

BB84 protocol complete — Alice and Bob share a secret key!
